# Pipeline

In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder

In [49]:
import pandas as pd 
df=pd.read_csv('titanic.csv')
df.drop(columns=['Name','SibSp','Ticket','Cabin','PassengerId'], inplace=True)
df.head()

,Survived,Pclass,Sex,Age,Parch,Fare,Embarked
0,0,3,male,22.0,0,7.2500,S
1,1,1,female,38.0,0,71.2833,C
2,1,3,female,26.0,0,7.9250,S
3,1,1,female,35.0,0,53.1000,S
4,0,3,male,35.0,0,8.0500,S


In [50]:
x=df.drop(columns=['Survived'])
y=df["Survived"]
x_train ,x_test, y_train , y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [73]:
preprocessing = ColumnTransformer(transformers=[
    ('age_fill',       SimpleImputer(),                          [2]),
    ('embarked_fill',  SimpleImputer(strategy='most_frequent'), [5])
], remainder='passthrough')
x=preprocessing.fit_transform(x_train)
x

array([[4.0, 'S', 1, 'male', 2, 81.8583],
       [29.256352705410823, 'S', 3, 'male', 0, 7.8958],
       [1.0, 'S', 3, 'female', 1, 11.1333],
       ...,
       [41.0, 'S', 3, 'male', 0, 14.1083],
       [14.0, 'S', 1, 'female', 2, 120.0],
       [21.0, 'S', 1, 'male', 1, 77.2875]], shape=(623, 6), dtype=object)

In [74]:
OHE_Embarked_sex=ColumnTransformer(transformers=[
    ('ohe_embarked', OneHotEncoder(),[1,3])
],remainder='passthrough')

In [75]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest , chi2
from sklearn.pipeline import Pipeline , make_pipeline

In [76]:
scaled=ColumnTransformer(transformers=[
    ('scale', MinMaxScaler(), slice(0,9))
])

In [77]:
Feature_selection= SelectKBest(score_func=chi2,k=7)

In [78]:
from sklearn import tree
clf = tree.DecisionTreeClassifier()

In [79]:
pipe= Pipeline([
    ('miss_em',preprocessing), 
    ('ohe', OHE_Embarked_sex), 
    ('scale',scaled), 
    ('featureselect',Feature_selection), 
    ('model',clf),
]
)

In [80]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('miss_em',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age_fill', SimpleImputer(),
                                                  [2]),
                                                 ('embarked_fill',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [5])])),
                ('ohe',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_embarked',
                                                  OneHotEncoder(), [1, 3])])),
                ('scale',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 9, None))])),
                ('featureselect',
                 SelectKBest(k=7,
                             score_func=<function chi2 at 0x00000261C31A99E0>)),
                ('model', DecisionTreeClassifier())])

In [81]:
# alternative 
pipe2=make_pipeline(preprocessing,OHE_Embarked_sex,scaled,Feature_selection,clf)
pipe2.fit(x_train,y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('age_fill', SimpleImputer(),
                                                  [2]),
                                                 ('embarked_fill',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [5])])),
                ('columntransformer-2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_embarked',
                                                  OneHotEncoder(), [1, 3])])),
                ('columntransformer-3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 9, None))])),
                ('selectkbest',
                 SelectKBest(k=7,
                             score_func=<function chi2 at 0x00000261C31A99E0>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [82]:
pipe.named_steps

{'miss_em': ColumnTransformer(remainder='passthrough',
                   transformers=[('age_fill', SimpleImputer(), [2]),
                                 ('embarked_fill',
                                  SimpleImputer(strategy='most_frequent'),
                                  [5])]),
 'ohe': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_embarked', OneHotEncoder(), [1, 3])]),
 'scale': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 9, None))]),
 'featureselect': SelectKBest(k=7, score_func=<function chi2 at 0x00000261C31A99E0>),
 'model': DecisionTreeClassifier()}

In [83]:
y_preed=pipe.predict(x_test)

In [85]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_preed)

0.7910447761194029

# cross validation using pipeline

In [90]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, x_train,y_train,cv=6, scoring='accuracy')

array([0.70192308, 0.83653846, 0.83653846, 0.76923077, 0.79807692,
       0.89320388])

In [91]:
cross_val_score(pipe, x_train,y_train,cv=6, scoring='accuracy').mean()

np.float64(0.8075211600697036)

# GridSearchCV hyperparameter tunning

In [105]:
from sklearn.model_selection import GridSearchCV
params={
    'model__max_depth':[1,2,3,4,5, None]
}

In [106]:
grid=GridSearchCV(pipe, params, cv=5,scoring='accuracy')
grid.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('miss_em',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('age_fill',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('embarked_fill',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [5])])),
                                       ('ohe',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_embarked',
                                                                         OneHotEncoder(),
                                                                         [1,
                                                                          3])])),
                                       ('scale',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 9, None))])),
                                       ('featureselect',
                                        SelectKBest(k=7,
                                                    score_func=<function chi2 at 0x00000261C31A99E0>)),
                                       ('model', DecisionTreeClassifier())]),
             param_grid={'model__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [107]:
grid.best_score_

np.float64(0.8010193548387097)

In [108]:
grid.best_params_

{'model__max_depth': None}

# exporting Pipeline

In [110]:
import pickle
pickle.dump(pipe,open("pipeline.pkl",'wb'))

# importing pipeline

In [112]:
import pickle
load_pipe=pickle.load(open('pipeline.pkl','rb'))

In [113]:
x_test.head(2)

,Pclass,Sex,Age,Parch,Fare,Embarked
709,3,male,NaN,1,15.2458,C
439,2,male,31.0,0,10.5000,S


In [118]:
test_input=np.array([2,'male',31,0,18.6, 'S'], dtype=object).reshape(1,6)
test_input

array([[2, 'male', 31, 0, 18.6, 'S']], dtype=object)

In [119]:
load_pipe.predict(test_input)

C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])